#Agent 1 – Segmentation Agent

Its job is to split the transcript into logical discussion sections.
Example:
Input:
Raj:
We need to launch the website by Friday.
 
Ankit:
Backend is completed.
 
Raj:
Let's assign testing to Rahul.
 
Rahul:
I'll finish it by Thursday.


Output:
Segment 1
Website launch discussion
 
Segment 2
Backend progress
 
Segment 3
Testing assignment

Agent 2 – Summarizer Agent
This agent summarizes every segment.
Example:
Segment 1
↓
Website launch is scheduled for Friday.
 
Segment 2
↓
Backend development has been completed.
 
Segment 3
↓
Rahul is responsible for testing and will finish by Thursday.


Agent 3 – Action Extraction Agent
This agent extracts structured tasks.
Example output:
Action
Owner
Due Date
Status
Finish testing
Rahul
Thursday
Pending
Launch website
Team
Friday
Pending

=========================================
Meeting Summary
=========================================
 
The team discussed the upcoming website launch scheduled for Friday.
Backend development has been completed.
Testing responsibilities were assigned to Rahul.
The team agreed to complete all pending work before launch.
 
=========================================
Action Items
=========================================
 
Action:
Complete Testing
 
Owner:
Rahul
 
Due:
Thursday
 
Status:
Pending

# Task 2 - Meeting Transcript to Summary and Action Items
 
## Objective
 
Build a multi-agent pipeline using LangGraph that converts a raw meeting transcript into:
 
- A concise meeting summary
- Structured action items
 
---
 
## Workflow
 
Meeting Transcript
        │
        ▼
Segmentation Agent
        │
        ▼
Summarizer Agent
        │
        ▼
Action Extraction Agent
        │
        ▼
Summary + Action Items

In [2]:
# ==========================================
# Step 1 : Import Required Libraries
# ==========================================
 
import os
from dotenv import load_dotenv
from typing import TypedDict
 
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic

In [3]:
# ==========================================
# Step 2 : Load Environment Variables
# ==========================================
 
load_dotenv(override=True)
 
llm = ChatAnthropic(
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_ENDPOINT"),
    temperature=0
)
 
print("LLM Loaded Successfully")

LLM Loaded Successfully


In [4]:
# ==========================================
# Step 3 : Define State
# ==========================================
 
class MeetingState(TypedDict):
 
    transcript: str
 
    segments: str
 
    summary: str
 
    actions: str

In [5]:
# ==========================================
# Step 4 : Sample Meeting Transcript
# ==========================================
 
meeting_transcript = """
Raj:
Good morning everyone.
Our website launch is planned for Friday.
 
Ankit:
Backend development is completed.
 
Priya:
Frontend integration will finish tomorrow.
 
Rahul:
I will complete testing by Thursday.
 
Raj:
Great. Rahul will send the testing report.
Priya will deploy the frontend.
"""

In [6]:
# ==========================================
# Step 5 : Segmentation Agent
# ==========================================
 
def segmentation_agent(state: MeetingState):
 
    prompt = f"""
You are a Meeting Segmentation Agent.
 
Divide the following meeting transcript into logical discussion sections.
 
Meeting Transcript:
 
{state["transcript"]}
 
Return clean sections only.
"""
 
    answer = llm.invoke(prompt)
 
    print("Meeting Segments Generated")
 
    state["segments"] = answer.content
 
    return state

## Step 6
 
Test the Segmentation Agent before building the graph.

In [7]:

#Step 6 : Test Segmentation Agent
sample = {
    "transcript": meeting_transcript
}
 
result = segmentation_agent(sample)
 
print(result["segments"])

Meeting Segments Generated
# Meeting Segmentation

## Section 1: Website Launch Overview
- Raj announces website launch planned for Friday

## Section 2: Development Status Updates
- Ankit reports backend development is completed
- Priya reports frontend integration will finish tomorrow

## Section 3: Testing & Deployment Plan
- Rahul commits to complete testing by Thursday
- Raj assigns Rahul to send testing report
- Raj assigns Priya to deploy the frontend


### Summarizer Agent
 
This agent reads the segmented meeting discussion and creates a concise meeting summary.
Step 7 : Summarizer Agent

In [8]:

# ==========================================
# Step 7 : Summarizer Agent
# ==========================================
 
def summarizer_agent(state: MeetingState):
 
    prompt = f"""
You are a Meeting Summarizer.
 
Below are the segmented meeting notes.
 
{state["segments"]}
 
Generate a concise meeting summary in 3-4 sentences.
"""
 
    answer = llm.invoke(prompt)
 
    print("Meeting Summary Generated")
 
    state["summary"] = answer.content
 
    return state

In [9]:
#Step 8 : Test Summarizer Agent
result = summarizer_agent(result)
 
print(result["summary"])

Meeting Summary Generated
# Meeting Summary

The team discussed the website launch scheduled for Friday. Ankit confirmed that backend development is complete, while Priya indicated frontend integration will be finished by tomorrow. Rahul committed to completing all testing by Thursday and will provide a testing report, with Priya assigned to handle the frontend deployment. The team is on track to meet the Friday launch deadline.


## Step 9
 
### Action Extraction Agent
 
This agent extracts tasks assigned during the meeting.
 
Each action item contains:
 
- Action Description
- Owner
- Due Date

In [10]:

# ==========================================
# Step 9 : Action Extraction Agent
# ==========================================
 
def action_agent(state: MeetingState):
 
    prompt = f"""
You are an Action Item Extraction Agent.
 
Using the meeting summary below,
 
{state["summary"]}
 
Extract all action items.
 
Return them as:
 
Action:
Owner:
Due Date:
"""
 
    answer = llm.invoke(prompt)
 
    print("Action Items Generated")
 
    state["actions"] = answer.content
 
    return state

In [11]:
#Step 10 : Test Action Agent
result = action_agent(result)
 
print(result["actions"])

Action Items Generated
# Action Items

**Action:** Complete frontend integration
**Owner:** Priya
**Due Date:** Tomorrow

---

**Action:** Complete all testing
**Owner:** Rahul
**Due Date:** Thursday

---

**Action:** Provide testing report
**Owner:** Rahul
**Due Date:** Thursday

---

**Action:** Handle frontend deployment
**Owner:** Priya
**Due Date:** Friday


In [ ]:
# ==========================================
# Step 11 : Build LangGraph
# ==========================================
 
builder = StateGraph(MeetingState)
 
builder.add_node("segment", segmentation_agent)
 
builder.add_node("summary", summarizer_agent)
 
builder.add_node("actions", action_agent)



 
builder.add_edge(START, "segment")
 
builder.add_edge("segment", "summary")
 
builder.add_edge("summary", "actions")
 
builder.add_edge("actions", END)
 
print("Graph Created")

Graph Created


In [13]:
#Step 12 : Compile Graph
graph = builder.compile()
 
print("Graph Compiled Successfully")

Graph Compiled Successfully


### Execute the Complete LangGraph Pipeline
 
This step runs the complete meeting-processing workflow:
 
1. Segment the transcript
2. Generate the meeting summary
3. Extract action items

In [14]:


# ==========================================
# Step 13 : Execute LangGraph
# ==========================================
 
result = graph.invoke(
    {
        "transcript": meeting_transcript,
        "segments": "",
        "summary": "",
        "actions": ""
    }
)
 
print("Pipeline Executed Successfully!")

Meeting Segments Generated
Meeting Summary Generated
Action Items Generated
Pipeline Executed Successfully!


In [15]:
### Display Final Meeting Summary
#Step 14 : Print Summary
print("=" * 60)
print("MEETING SUMMARY")
print("=" * 60)
 
print(result["summary"])

MEETING SUMMARY
# Meeting Summary

The team discussed the website launch scheduled for Friday. Ankit confirmed that backend development is complete, while Priya indicated frontend integration will be finished by tomorrow. Rahul committed to completing all testing by Thursday and will provide a testing report, with Priya assigned to handle the frontend deployment. The team is on track to meet the Friday launch deadline.


In [16]:
## Step 15
 
### Display Extracted Action Items
#Step 15 : Print Action Items
print("=" * 60)
print("ACTION ITEMS")
print("=" * 60)
 
print(result["actions"])

ACTION ITEMS
# Action Items

**Action:** Complete frontend integration
**Owner:** Priya
**Due Date:** Tomorrow

---

**Action:** Complete all testing
**Owner:** Rahul
**Due Date:** Thursday

---

**Action:** Provide testing report
**Owner:** Rahul
**Due Date:** Thursday

---

**Action:** Handle frontend deployment
**Owner:** Priya
**Due Date:** Friday


# LangGraph Workflow
 
```mermaid
graph TD
 
START --> SegmentationAgent
 
SegmentationAgent --> SummarizerAgent
 
SummarizerAgent --> ActionExtractionAgent
 
ActionExtractionAgent --> END
```
 
This LangGraph workflow processes a meeting transcript by:
 
- Splitting the transcript into logical discussion segments.
- Generating a concise meeting summary.
- Extracting structured action items including owners and due dates.

In [18]:
print("=" * 60)
print("TRANSCRIPT")
print("=" * 60)
print(result["transcript"])
 
print("\n" + "=" * 60)
print("SEGMENTS")
print("=" * 60)
print(result["segments"])
 
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(result["summary"])
 
print("\n" + "=" * 60)
print("ACTION ITEMS")
print("=" * 60)
print(result["actions"])

TRANSCRIPT

Raj:
Good morning everyone.
Our website launch is planned for Friday.

Ankit:
Backend development is completed.

Priya:
Frontend integration will finish tomorrow.

Rahul:
I will complete testing by Thursday.

Raj:
Great. Rahul will send the testing report.
Priya will deploy the frontend.


SEGMENTS
# Meeting Segmentation

## Section 1: Website Launch Overview
- Raj announces website launch planned for Friday

## Section 2: Development Status Updates
- Ankit reports backend development is completed
- Priya reports frontend integration will finish tomorrow

## Section 3: Testing & Deployment Plan
- Rahul commits to complete testing by Thursday
- Raj assigns Rahul to send testing report
- Raj assigns Priya to deploy the frontend

SUMMARY
# Meeting Summary

The team discussed the website launch scheduled for Friday. Ankit confirmed that backend development is complete, while Priya indicated frontend integration will be finished by tomorrow. Rahul committed to completing all tes